# DenseNet121 — Group images without labels

Use case: organize an unlabeled photo collection into groups. DenseNet reuses earlier feature maps through connections to later layers. K-means groups similar feature vectors without training on category labels.

Run the cells from top to bottom. Each notebook is independent.


## Step 1 — Install packages
Use Python 3.11 or 3.12 as a straightforward training environment, or Google Colab. Install into the active notebook kernel. Restart the kernel after the first installation if required. Internet is needed initially for packages, sample images/data and ImageNet weights. A GPU is optional; CPU runs can be slower. Model downloads can be hundreds of MB.


In [ ]:
%pip install tensorflow numpy matplotlib scikit-learn pillow


## Step 2 — Import libraries
TensorFlow runs the neural network. Keras provides the model API. NumPy handles arrays and Matplotlib displays pictures.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

keras.utils.set_random_seed(42)
print("TensorFlow:", tf.__version__)


## Step 3 — Load a small image collection
We use 300 CIFAR-10 images. Labels are retained only for interpretation after clustering.


In [ ]:
(train_images, train_labels), (test_images, test_labels) = keras.datasets.cifar10.load_data()
train_labels = train_labels.ravel()
test_labels = test_labels.ravel()
class_names = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]
print("Training images:", train_images.shape)
print("Test images:", test_images.shape)
plt.figure(figsize=(9, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(train_images[i])
    plt.title(class_names[train_labels[i]])
    plt.axis("off")
plt.tight_layout()
plt.show()


## Step 4 — Extract pretrained features
Pooling creates one feature vector per image. We normalize it before clustering.


In [ ]:
from tensorflow.keras.applications.densenet import DenseNet121, preprocess_input
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
images = train_images[:300]
extractor = DenseNet121(weights="imagenet", include_top=False,
                        pooling="avg", input_shape=(224, 224, 3))
parts = []
for start in range(0, len(images), 16):
    batch = tf.image.resize(images[start:start + 16], (224, 224))
    parts.append(extractor.predict(preprocess_input(batch), verbose=0))
features = np.concatenate(parts)
features = features / np.maximum(np.linalg.norm(features, axis=1, keepdims=True), 1e-8)


## Step 5 — Create clusters
A cluster ID is an arbitrary group number, not a class name. We choose five groups for exploration.


In [ ]:
clusterer = KMeans(n_clusters=5, random_state=42, n_init=10)
cluster_ids = clusterer.fit_predict(features)
for group in range(5):
    print("Cluster", group, "images:", np.sum(cluster_ids == group))


## Step 6 — Inspect representative group members
Mixed labels within a cluster can occur because visual appearance and semantic categories differ.


In [ ]:
for group in range(5):
    indices = np.where(cluster_ids == group)[0][:6]
    plt.figure(figsize=(10, 2))
    for position, index in enumerate(indices, start=1):
        plt.subplot(1, 6, position)
        plt.imshow(images[index])
        plt.title(class_names[train_labels[index]])
        plt.axis("off")
    plt.suptitle(f"Cluster {group}")
    plt.tight_layout()
    plt.show()


## Step 7 — Visualize the embedding collection
PCA compresses vectors to two dimensions for plotting. The plot loses information and is not the space used by K-means.


In [ ]:
points = PCA(n_components=2, random_state=42).fit_transform(features)
plt.scatter(points[:, 0], points[:, 1], c=cluster_ids, cmap="tab10", s=20)
plt.xlabel("Principal component 1")
plt.ylabel("Principal component 2")
plt.title("Image clusters: 2D projection")
plt.show()


## Understand the result
This is unsupervised grouping. There is no prediction of the ten CIFAR category names. Try changing n_clusters and visually inspect whether the groups are useful.


## Practice
1. Change the input image or selected image index.
2. Explain the output in your own words.
3. Compare the result with the original image and record one limitation.


## References
- [Keras Applications](https://keras.io/api/applications/)
- [CIFAR-10 dataset](https://www.cs.toronto.edu/~kriz/cifar.html)
- [Keras Grad-CAM example](https://keras.io/examples/vision/grad_cam/)
